## Notebook の初期設定

共通の表示スタイルを読み込み、以降の DataFrame 表示を本文用の体裁に整える。


In [14]:
import sys
from pathlib import Path

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path))
        break

    if (path / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "src"))
        break

from common.notebook_style import apply_dataframe_style

apply_dataframe_style()

# 補足：Aprioriアルゴリズムのスクラッチ実装

## Apriori アルゴリズムのスクラッチ実装（頻出アイテムセットの探索）

本文ではマーケティング上の解釈を優先し、スクラッチ実装の詳細はこの Notebook の補足として扱う。
ここでは mlxtend を使わずに Apriori アルゴリズムを実装し、頻出アイテムセットを探索する。
TIDリスト（ハッシュテーブル）と枝刈りを用いることで、支持度計算と候補削減の仕組みを確認する。

### 処理内容

1. **パラメータの設定**
   最小支持度 `min_support=0.005`、アイテムセット最小サイズ `min_comb=2`、最大サイズ `max_comb=6` を設定する。ここでは `2 <= min_comb <= max_comb` を対象とする。

2. **ワンホットエンコーディング（sklearn）**
   `MultiLabelBinarizer` でトランザクションデータを数値行列に変換し、Polars DataFrame に格納する。

3. **TIDリスト（ハッシュテーブル）の構築**
   各アイテムが出現するトランザクションのインデックス集合を `frozenset` キーの辞書 `item_set2indices` に格納する。
   これによりトランザクション全体を毎回走査せず、集合の積集合によって支持度を計算できる。

4. **1-アイテムセットのフィルタリング**
   出現率が `min_support` 以上のアイテムのみを `initial_set` として抽出する。

5. **段階的なアイテムセット拡張と枝刈り**
   サイズ2から `max_comb` まで候補を拡張し、TIDリストの積集合で支持度を計算する。出力にはサイズが `min_comb` 以上の集合だけを記録する。
   Apriori 性質（頻出アイテムセットのすべての部分集合も必ず頻出）に基づき、
   最小支持度を下回るアイテムセットを探索から除外する。
   説明を簡単にするため、候補のすべての部分集合が頻出かを先に調べる処理と、非頻出候補の再計算を避ける処理は省いている。


In [16]:
import polars as pl

# ファイルの指定
file_name = "./data/Groceries data.csv"

# CSVを読み込み、使う3列を選択
df = pl.read_csv(file_name).select(["Member_number", "Date", "itemDescription"])

print(f"行数: {df.height}, 列数: {df.width}")

### トランザクション形式への整形

同一会員・同一購入日の商品明細を集約し、購入商品をリストとしてまとめたトランザクション形式（リストのリスト）に変換する。

#### 処理内容

1. **グループ化と集約**
   `df.group_by(["Member_number", "Date"])` で会員番号と購入日をキーにグループ化し、
   `pl.col("itemDescription").unique(maintain_order=True)` で商品名を重複排除してリスト化する。

2. **トランザクションリストの抽出**
   `transactions_df["items"].to_list()` で商品リストだけを取り出し、リストのリストに変換する。

3. **先頭データの確認**
   `transactions[:5]` で先頭5件のトランザクションを表示する。


In [18]:
# 同じ会員番号・同じ購入日の商品明細を1つの買い物として集約
transactions_df = df.group_by(
    ["Member_number", "Date"],
    maintain_order=True,
).agg(pl.col("itemDescription").unique(maintain_order=True).alias("items"))

# 商品リストだけを取り出して、リストのリストに変換
transactions = transactions_df["items"].to_list()

# 最初の5行を表示
transactions[:5]

[['tropical fruit', 'rolls/buns', 'candy'],
 ['whole milk', 'tropical fruit', 'chocolate'],
 ['pip fruit', 'other vegetables', 'flour'],
 ['other vegetables', 'onions', 'shopping bags'],
 ['whole milk', 'other vegetables', 'white bread']]

In [19]:
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer

# 最小支持度の閾値設定
min_support = 0.005

# 最小アイテムセットサイズの設定
min_comb = 2

# 最大アイテムセットサイズの設定
max_comb = 6

# ワンホットエンコーディング処理
mlb = MultiLabelBinarizer()
encoded = mlb.fit_transform(transactions)
df = pl.DataFrame(encoded, schema=list(mlb.classes_)).with_row_index("row_nr")
item_columns = [c for c in df.columns if c != "row_nr"]

# トランザクションデータ数の取得
transactions_length = df.height

# 1-アイテムセットの集合から行番号へのハッシュテーブルの作成
# アイテムセットからTIDリストを直接参照できるようにハッシュテーブル（辞書型）とする
# 通常のset型は辞書のキーにはできないが、frozensetを利用することでキーとして利用する
item_set2indices = {
    frozenset({c}): frozenset(df.filter(pl.col(c) == 1)["row_nr"].to_list())
    for c in item_columns
}

# 最小支持度以上の1-アイテムセットの抽出
initial_set = {
    frozenset({c})
    for c in item_columns
    if df.select(pl.col(c).mean()).item() >= min_support
}

# current_setという名前で、条件を満たす組み合わせアイテムの集合を格納する
# current_setに、随時条件を満たすアイテムセットが追加される
current_set = initial_set

# 最小支持度以上の組み合わせを格納するリスト
frequent_item_list = []

# 最小支持度以上の組み合わせの支持度を格納するリスト
support_list = []

# サイズ2から最大アイテムセットサイズまで候補を拡張する
# min_comb は出力に含める最小サイズとして使う
for _ in range(2, max_comb + 1):
    # set を使うことで重複した frozenset の登録を防ぐ
    next_set = set({})

    for _c in current_set:
        for _i in initial_set:
            # initial_setから取りだした要素が、current_setの要素の一部(サブセット)になっているか判定
            # サブセットでない場合は、そのor集合のfrozensetが item_set2indices に含まれていないか確認
            if (
                not _i.issubset(_c)
                and frozenset(_c | _i) not in item_set2indices
            ):
                _support = (
                    len(item_set2indices[_i] & item_set2indices[_c])
                    / transactions_length
                )

                # 和集合の支持度が最小支持度以上の場合、共起頻度が高いアイテムセットと認め、frequent_item_listに追記。その支持度値も追記。
                if _support >= min_support:
                    if len(_c | _i) >= min_comb:
                        frequent_item_list.append(frozenset(_c | _i))
                        support_list.append(_support)

                    # 和集合のindexの値をハッシュテーブルに追記
                    # そのindex自体は元の集合のindexの積集合であることに注意
                    item_set2indices[frozenset(_c | _i)] = (
                        item_set2indices[_i] & item_set2indices[_c]
                    )

                    # この和集合が次のcurrent_setになる
                    next_set.add(frozenset(_c | _i))

    if next_set:
        # next_setをcurrent_setにして次の組み合わせを抽出する
        current_set = next_set

        # current_setの構成要素を分解し1-アイテムセットに戻す
        # 頻出k-アイテムセットに含まれるアイテムのみに initial_set を絞ることで
        # 不要な候補生成を排除し（枝刈り）、探索空間を削減する
        next_initial_set = set({})
        for _c in current_set:
            for __c in _c:
                next_initial_set.add(frozenset({__c}))
        initial_set = frozenset(next_initial_set)

    else:
        # next_setが空のままの場合 break
        break

# 補足：Aprioriスクラッチ実装へのルール適用

## Aprioriスクラッチ実装へのルール適用

スクラッチ実装で得た `frequent_item_list` と `support_list` に対して、
確信度とリフト値を計算しアソシエーションルールを生成する。
mlxtend による結果と同一のルールが得られるかを確認する。

### 処理内容

1. **閾値の設定**
   最小リフト値 `min_lift=1.0`、最小確信度 `min_confidence=0.02` を設定する。

2. **アソシエーションルールの生成**
   各頻出アイテムセットについて、1アイテムを結論部・残りを条件部とするルールを総当たりで検討する。
   TIDリストを用いて条件部・結論部の支持度を計算し、確信度とリフト値を求める。

3. **フィルタリングと結果整理**
   閾値を満たすルールを Polars DataFrame にまとめ、
   支持度・確信度・リフト値の降順でソートし小数点4桁で表示する。


In [20]:
# 閾値となる最小確信度と最小リフト値の設定
min_lift = 1.0
min_confidence = 0.02

# 条件を満たしたルールを格納するリスト
association_item_list = []

# frequent_item_listとその支持度値を同時に取り出す
for item_set, support in zip(frequent_item_list, support_list):
    if len(item_set) > 1:
        for item in item_set:
            antecedent = item_set - {item}

            # 条件部の支持度を計算
            antecedent_support = (
                len(item_set2indices[antecedent]) / transactions_length
            )

            # 結論部の支持度を計算
            consequence_support = (
                len(item_set2indices[frozenset({item})]) / transactions_length
            )

            if antecedent_support > 0:
                # 確信度の計算
                # Conf(X => Y) = P(Y|X) = P(X&Y) / P(X)
                confidence = support / antecedent_support

                # リフト値の計算
                # Lift(X => Y) = P(X&Y) / P(X) / P(Y) = Conf(X => Y) / P(Y)
                lift = confidence / consequence_support

                # 条件を満たすか確認
                if (confidence >= min_confidence) and (lift >= min_lift):
                    consequence = {item}
                    association_item_list.append(
                        (antecedent, consequence, support, confidence, lift)
                    )

# 表示するカラム名の設定
columns = ["antecedents", "consequents", "support", "confidence", "lift"]

# Polars DataFrame型で読み込み、support、confidence、liftの順番で降順にソート
result_df = pl.DataFrame(
    [
        {
            "antecedents": sorted(antecedent),
            "consequents": sorted(consequence),
            "support": support,
            "confidence": confidence,
            "lift": lift,
        }
        for antecedent, consequence, support, confidence, lift in association_item_list
    ]
).sort(["support", "confidence", "lift"], descending=[True, True, True])

# 後段の比較で利用するために保存
apriori_result_df = result_df.clone()

# 小数点4桁で表示
result_df.with_columns(
    pl.col(["support", "confidence", "lift"]).round(4)
).head()

antecedents,consequents,support,confidence,lift
list[str],list[str],f64,f64,f64
"[""sausage""]","[""soda""]",0.0059,0.0986,1.015
"[""soda""]","[""sausage""]",0.0059,0.0613,1.015
"[""sausage""]","[""yogurt""]",0.0057,0.0952,1.109
"[""yogurt""]","[""sausage""]",0.0057,0.0669,1.109
"[""frankfurter""]","[""other vegetables""]",0.0051,0.1363,1.1161


# 補足：Eclatアルゴリズムのスクラッチ実装

## Eclat アルゴリズムの実装

Eclat（Equivalence Class Transformation）アルゴリズムをスクラッチで実装し、頻出アイテムセットを抽出する。
Apriori とは異なり、深さ優先探索（DFS）と TIDリストの積集合を用いて頻出アイテムセットを探索する。

### 処理内容

1. **パラメータ・前処理**
   Apriori スクラッチ実装と同じパラメータ設定・TIDリスト構築・1-アイテムセットフィルタリングを行う。

2. **`eclat` 関数の定義（深さ優先探索）**
   候補リストからアイテムセットをポップし、TIDリストの積集合で支持度を計算する。
   支持度が `min_support` 以上であれば頻出として記録し、
   残り候補と積集合が閾値を超えるものを次候補として再帰呼び出しする。

3. **Eclat の実行**
   TIDリストのサイズ降順に並べた全アイテムを初期候補として `eclat()` を呼び出し、頻出アイテムセットを収集する。

> **注**: Eclat の結果に対するアソシエーションルールの生成は、次のセルで個別に実行する。


In [22]:
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer

# 最小支持度の閾値設定
min_support = 0.005

# 最小アイテムセットサイズの設定
min_comb = 2

# 最大アイテムセットサイズの設定
max_comb = 6

# ワンホットエンコーディング処理
mlb = MultiLabelBinarizer()
encoded = mlb.fit_transform(transactions)
df = pl.DataFrame(encoded, schema=list(mlb.classes_)).with_row_index("row_nr")
item_columns = [c for c in df.columns if c != "row_nr"]

# トランザクションデータ数の取得
transactions_length = df.height

# 1-アイテムセットの集合から行番号へのハッシュテーブルの作成
# アイテムセットからTIDリストを直接参照できるようにハッシュテーブル（辞書型）とする
# 通常のset型は辞書のキーにはできないが、frozensetを利用することでキーとして利用する
item_set2indices = {
    frozenset({c}): frozenset(df.filter(pl.col(c) == 1)["row_nr"].to_list())
    for c in item_columns
}

# 最小支持度以上の1-アイテムセットの抽出
initial_set = {
    frozenset({c})
    for c in item_columns
    if df.select(pl.col(c).mean()).item() >= min_support
}

current_set = initial_set

# 最小支持度以上の組み合わせを格納するリスト
frequent_item_list = []

# 最小支持度以上の組み合わせの支持度を格納するリスト
support_list = []


def eclat(
    item_set2indices_items,
    selected_item_set,
    min_support,
    min_comb,
    max_comb,
    frequent_item_list,
    support_list,
):
    while item_set2indices_items:
        pop_item_set, pop_item_set_indices = item_set2indices_items.pop()
        candidate_item_set = frozenset(selected_item_set | pop_item_set)

        _support = len(pop_item_set_indices) / transactions_length
        if _support >= min_support:
            if len(candidate_item_set) >= min_comb:
                frequent_item_list.append(candidate_item_set)
                support_list.append(_support)

            # 和集合のindexの値をハッシュテーブルに追記
            # そのindex自体は元の集合のindexの積集合であることに注意
            if selected_item_set:
                item_set2indices[candidate_item_set] = (
                    item_set2indices[selected_item_set]
                    & item_set2indices[pop_item_set]
                )

            next_items_list = []
            if len(candidate_item_set) < max_comb:
                for _item_set, _item_set_indices in item_set2indices_items:
                    _support = (
                        len(pop_item_set_indices & _item_set_indices)
                        / transactions_length
                    )

                    if _support >= min_support:
                        next_items_list.append(
                            (
                                _item_set,
                                frozenset(
                                    pop_item_set_indices & _item_set_indices
                                ),
                            )
                        )

            eclat(
                sorted(next_items_list, key=lambda x: len(x[1]), reverse=True),
                candidate_item_set,
                min_support,
                min_comb,
                max_comb,
                frequent_item_list,
                support_list,
            )


selected_item_set = frozenset({})

eclat(
    sorted(item_set2indices.items(), key=lambda x: len(x[1]), reverse=True),
    selected_item_set,
    min_support,
    min_comb,
    max_comb,
    frequent_item_list,
    support_list,
)

## Eclat 結果へのアソシエーションルール適用

Eclat で抽出した頻出アイテムセットに対して、確信度とリフト値を計算し、ルール一覧を作成する。


In [23]:
# 閾値となる最小確信度と最小リフト値の設定
min_lift = 1.0
min_confidence = 0.02

# 条件を満たしたルールを格納するリスト
association_item_list = []

# frequent_item_listとその支持度値を同時に取り出す
for item_set, support in zip(frequent_item_list, support_list):
    if len(item_set) > 1:
        for item in item_set:
            antecedent = item_set - {item}

            # 条件部の支持度を計算
            antecedent_support = (
                len(item_set2indices[antecedent]) / transactions_length
            )

            # 結論部の支持度を計算
            consequence_support = (
                len(item_set2indices[frozenset({item})]) / transactions_length
            )

            if antecedent_support > 0:
                # 確信度の計算
                confidence = support / antecedent_support

                # リフト値の計算
                lift = confidence / consequence_support

                # 条件を満たすか確認
                if (confidence >= min_confidence) and (lift >= min_lift):
                    consequence = {item}
                    association_item_list.append(
                        (antecedent, consequence, support, confidence, lift)
                    )

# 表示するカラム名の設定
columns = ["antecedents", "consequents", "support", "confidence", "lift"]

# Polars DataFrame型で読み込み、support、confidence、liftの順番で降順にソート
result_df = pl.DataFrame(
    [
        {
            "antecedents": sorted(antecedent),
            "consequents": sorted(consequence),
            "support": support,
            "confidence": confidence,
            "lift": lift,
        }
        for antecedent, consequence, support, confidence, lift in association_item_list
    ]
).sort(["support", "confidence", "lift"], descending=[True, True, True])

# 後段の比較で利用するために保存
eclat_result_df = result_df.clone()

# 小数点4桁で表示
result_df.with_columns(
    pl.col(["support", "confidence", "lift"]).round(4)
).head()

antecedents,consequents,support,confidence,lift
list[str],list[str],f64,f64,f64
"[""sausage""]","[""soda""]",0.0059,0.0986,1.015
"[""soda""]","[""sausage""]",0.0059,0.0613,1.015
"[""sausage""]","[""yogurt""]",0.0057,0.0952,1.109
"[""yogurt""]","[""sausage""]",0.0057,0.0669,1.109
"[""frankfurter""]","[""other vegetables""]",0.0051,0.1363,1.1161


## mlxtend による計算


In [25]:
import polars as pl

# 全アイテムの一覧を取得
all_items = sorted(
    {item for transaction in transactions for item in transaction}
)

# 各アイテムをカラムに設定し、各トランザクションでアイテムの購入有無をbool型で格納
# ワンホットエンコーディング処理
df = pl.DataFrame(
    {
        item: [item in transaction for transaction in transactions]
        for item in all_items
    }
)

preview_columns = (
    transactions[0]
    + [item for item in all_items if item not in transactions[0]][:4]
)
df.slice(0, 5).select(preview_columns)

# apriori モジュールの読み込み
from mlxtend.frequent_patterns import apriori

# mlxtend は pandas DataFrame を要求するため変換
df_pd = df.to_pandas()

# 共起性の高いアイテムセットの抽出
frequent_itemsets = apriori(df_pd, min_support=0.005, use_colnames=True)
frequent_itemsets.head()


from mlxtend.frequent_patterns import association_rules

min_threshold = 0.02

# 頻繁に購入されるアイテムセット（frequent_itemsets）から、Confidenceが0.02以上のものを抽出
rules_df = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_threshold,
)

# リフト値が1.0以上のルールに絞り込む
rules_df = rules_df.query("lift >= 1.0")

# 利用するカラム名の設定
columns = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
]

# 後段の比較で利用するために保存
mlxtend_result_df = rules_df[columns].copy()

# support, confidence, lift の順番に降順でソートして小数点4桁で表示
rules_df.sort_values(by=["support", "confidence", "lift"], ascending=False)[
    columns
].reset_index(drop=True).round(4).head()

,antecedents,consequents,support,confidence,lift
0,frozenset({sausage}),frozenset({soda}),0.0059,0.0986,1.0150
1,frozenset({soda}),frozenset({sausage}),0.0059,0.0613,1.0150
2,frozenset({sausage}),frozenset({yogurt}),0.0057,0.0952,1.1090
3,frozenset({yogurt}),frozenset({sausage}),0.0057,0.0669,1.1090
4,frozenset({frankfurter}),frozenset({other vegetables}),0.0051,0.1363,1.1161


## 3手法のルール集合比較

本文で用いた mlxtend の結果と、補足として実装した Apriori・Eclat のスクラッチ実装で得られたルール集合が一致するかを確認する。


In [26]:
def normalize_itemset(itemset):
    return tuple(sorted(itemset))


def normalize_rules(df):
    records = (
        df.select(columns).to_dicts()
        if isinstance(df, pl.DataFrame)
        else df[columns].to_dict("records")
    )
    return {
        (
            normalize_itemset(record["antecedents"]),
            normalize_itemset(record["consequents"]),
            round(float(record["support"]), 10),
            round(float(record["confidence"]), 10),
            round(float(record["lift"]), 10),
        )
        for record in records
    }


rule_sets = {
    "mlxtend": normalize_rules(mlxtend_result_df),
    "apriori_scratch": normalize_rules(apriori_result_df),
    "eclat_scratch": normalize_rules(eclat_result_df),
}

comparison_df = pl.DataFrame(
    {
        "method": list(rule_sets.keys()),
        "rule_count": [len(rule_set) for rule_set in rule_sets.values()],
        "same_as_mlxtend": [
            rule_set == rule_sets["mlxtend"] for rule_set in rule_sets.values()
        ],
    }
)
comparison_df

method,rule_count,same_as_mlxtend
str,i64,bool
"""mlxtend""",6,true
"""apriori_scratch""",6,true
"""eclat_scratch""",6,true
